# The PageRank Algorithm: Eigenvectors in Action
### A Linear Algebra Lab for Math 107

---

Welcome! In this lab you'll explore one of the most famous applications of linear algebra ever built: the **PageRank algorithm**, the mathematical idea behind Google's original search engine.

Every step is explained, and you'll mostly just be **modifying existing code** rather than writing it from scratch.

The notebook has a few custom-written functions. You are **not** responsible for understanding that code. Just run the cell and then you will be able to call the function in subsequent cells.

**How to use this notebook:**
- Each gray box (called a **cell**) contains either explanation text or Python code.
- To **run a code cell**, click on it and press **Shift + Enter** (or click the ▶ button in the toolbar).
- Run cells **in order from top to bottom** — this is important! Later cells use functions and variables defined in earlier cells, so if you skip a cell or run them out of order you may get an error. If something is not working, try going back to the top and running every cell in order again.
- Don't be afraid to experiment! You can always re-run a cell after changing it.


The notebook covers the following topics in order:

- The big idea: what is PageRank?
- Modeling the web as a graph and a matrix
- The transition matrix (a column-stochastic matrix)
- Power iteration: simulating a random surfer
- Eigenvalues and eigenvectors: why PageRank is an eigenvector problem
- Computing the exact PageRank with `np.linalg.eig`
- Practice problems

---

## Step 0: Setting Up — Run This First!

Before anything else, we need to import **NumPy** and **Matplotlib**, two powerful Python libraries for mathematics and plotting. Think of this as opening your math toolbox.

In [ ]:
# Lines that start with # are "comments" — Python ignores them.
# They're just notes for humans reading the code!

import numpy as np                  # NumPy: our main tool for matrices and vectors
import matplotlib.pyplot as plt     # Matplotlib: for making plots and graphs
import matplotlib.patches as mpatches

# This sets how many decimal places are displayed
np.set_printoptions(precision=4, suppress=True)

print("Libraries loaded successfully! You're ready to go.")

---

## Part 1: The Big Idea — What Is PageRank?

In the late 1990s, Larry Page and Sergey Brin (founders of Google) had a problem: when you search for something, how do you decide which web pages are *important* and should appear at the top of the results?

Their key insight: **a page is important if other important pages link to it.**

This is circular on the surface; "important" is defined in terms of "important," but linear algebra gives us a beautiful way to resolve it.

Here's the intuition with a simple **toy example**. Imagine a tiny internet with just **4 web pages** (A, B, C, D) and the following links between them:

```
  Page A links to: B, C
  Page B links to: C
  Page C links to: A
  Page D links to: A, B, C
```


Run the cell below to draw the graph automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_web_graph(links, title="Our Tiny Web"):
    pages = sorted(links.keys())
    n = len(pages)
    angles = [2 * np.pi * i / n - np.pi / 2 for i in range(n)]  # start from top
    pos = {p: (np.cos(a), np.sin(a)) for p, a in zip(pages, angles)}

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

    colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759']
    color_map = {p: colors[i] for i, p in enumerate(pages)}

    node_radius = 0.18  # controls circle size & arrow gap

    for src, targets in links.items():
        for tgt in targets:
            x0, y0 = pos[src]
            x1, y1 = pos[tgt]
            dx = x1 - x0
            dy = y1 - y0
            dist = np.hypot(dx, dy)
            # start/end at circle edges, not centers
            ux, uy = dx / dist, dy / dist
            sx = x0 + ux * node_radius
            sy = y0 + uy * node_radius
            ex = x1 - ux * node_radius
            ey = y1 - uy * node_radius
            ax.annotate(
                "", xy=(ex, ey), xytext=(sx, sy),
                arrowprops=dict(
                    arrowstyle='->', color='gray',
                    lw=1.8, connectionstyle='arc3,rad=0.1'
                )
            )

    for p in pages:
        x, y = pos[p]
        circle = plt.Circle((x, y), node_radius, color=color_map[p], zorder=3)
        ax.add_patch(circle)
        ax.text(x, y, p, ha='center', va='center',
                fontsize=16, fontweight='bold', color='white', zorder=4)

    plt.tight_layout()
    plt.show()

links = {
    'A': ['B', 'C'],
    'B': ['C'],
    'C': ['A'],
    'D': ['A', 'B', 'C']
}
draw_web_graph(links, title="Our Tiny 4-Page Web")

---

## Part 2: Modeling the Web with a Matrix

### 2.1 The Link Matrix

To do any linear algebra, we need to turn our graph into a **matrix**.

We'll consider the pages in order: (A, B, C, D).

We build a matrix $L$ where:
$$L_{ij} = 1 \quad \text{if page } j \text{ links to page } i, \quad 0 \text{ otherwise}$$

In other words, **column $j$ tells you which pages page $j$ points to**, and **row $i$ tells you which pages point to page $i$**.

For our tiny web:

|        | **from A** | **from B** | **from C** | **from D** |
|--------|-----------|-----------|-----------|----------|
| **to A** | 0         | 0         | 1         | 1        |
| **to B** | 1         | 0         | 0         | 1        |
| **to C** | 1         | 1         | 0         | 1        |
| **to D** | 0         | 0         | 0         | 0        |

Notice that page D has no pages linking *to* it ("to D" row is all zeros). Poor page D!

In [ ]:
# Build the link matrix L.
# Rows = destination page, Columns = source page
# Order: A=0, B=1, C=2, D=3

L = np.array([
    #  A  B  C  D
    [  0, 0, 1, 1 ],   # row A: who links TO A?
    [  1, 0, 0, 1 ],   # row B: who links TO B?
    [  1, 1, 0, 1 ],   # row C: who links TO C?
    [  0, 0, 0, 0 ],   # row D: who links TO D?
], dtype=float)

print("Link matrix L:")
print(L)
print()
print("Rows = destination page (A, B, C, D)")
print("Cols = source page      (A, B, C, D)")

### 2.2 The Transition Matrix (Column-Stochastic)

The link matrix $L$ is a good start, but we need to refine it.

**The random surfer model:** Imagine a user who clicks links at random. If they're on page A (which has 2 outgoing links, to B and C), they have a $\frac{1}{2}$ chance of going to B and a $\frac{1}{2}$ chance of going to C.

So we **normalize each column** by dividing by the number of outgoing links from that page. This turns $L$ into the **transition matrix** $M$:

$$M_{ij} = \frac{L_{ij}}{\text{(number of links out of page } j)}$$

After normalization, every **column of $M$ sums to 1**. This kind of matrix is called **column-stochastic**.

Each entry $M_{ij}$ is the *probability* that a random surfer on page $j$ jumps to page $i$ in one click.

In [ ]:
# Normalize each column of L by dividing by the column sum.
# This gives us the transition matrix M.

# Step 1: compute the sum of each column
col_sums = L.sum(axis=0)   # axis=0 means "sum down the rows" (i.e., sum each column)
print("Number of outgoing links per page (column sums):")
print("  A B C D")
print(" ", col_sums)
print()

# Step 2: divide each column by its sum
M = L / col_sums   # NumPy divides element-wise, broadcasting across rows

print("Transition matrix M (each column sums to 1):")
print(M)
print()
print("Column sums of M (should all be 1.0):")
print(M.sum(axis=0))

Take a moment to read that transition matrix $M$. Column A (the first column) says:
- Probability 0 of going to A (no self-link)
- Probability 0.5 of going to B
- Probability 0.5 of going to C
- Probability 0 of going to D

This matches the graph: A links to B and C, so a user could choose those pages with equal probability.

---

## Part 3: Power Iteration: Simulating the Random Surfer

### 3.1 What is a Rank Vector?

Heads up: there are two meanings of "rank."

- In linear algebra, the rank of a matrix measures how many linearly independent rows or columns it has. That's not what "rank" means here. 
- In PageRank, rank just means importance score (a popularity number we assign to each page). Same word, completely different concept. We'll always say rank vector or rank score to mean the importance score.


We want to assign a **rank score** to each page: a number between 0 and 1 that represents how important the page is. We'll store these in a vector:

$$\mathbf{r} = \begin{bmatrix} r_A \\ r_B \\ r_C \\ r_D \end{bmatrix}$$

The scores should sum to 1 (they represent probabilities).

**The key rule:** The rank of a page is the sum of the ranks of all pages that link to it, weighted by how many outgoing links those pages have. In matrix form:

$$\mathbf{r} = M \mathbf{r}$$

We want to find a rank vector $\mathbf{r}$ that **doesn't change** when we multiply by $M$. This is called a **steady state**.


**Let's unpack that.** Take page C in our example. It is linked to by A, B, and D, so the rule says:

$$r_C = \frac{r_A}{(\text{links out of A})} + \frac{r_B}{(\text{links out of B})} + \frac{r_D}{(\text{links out of D})} = \frac{r_A}{2} + \frac{r_B}{1} + \frac{r_D}{3}$$

Why divide by the number of outgoing links? Think of each page as having one "vote" worth of importance to give away, equal to its rank. If page A has rank $r_A$ and links to 2 pages, it splits that rank in half and sends $\frac{r_A}{2}$ to each page it points to. So a page's rank gets divided evenly among its outgoing links.

**Why is this the same as $\mathbf{r} = M\mathbf{r}$?** Look at what matrix-vector multiplication actually computes. To get the C-th entry of $M\mathbf{r}$, we take row C of $M$ and combine it with $\mathbf{r}$ by multiplying matching entries and adding them up.

Row C of $M$ tells us, for each page, the share of rank that page sends to C. Reading across row C of our transition matrix, the entries are $\frac{1}{2}, 1, 0, \frac{1}{3}$ (the contributions from A, B, C, and D). Multiplying each by the corresponding entry of $\mathbf{r}$ and adding gives:

$$(M\mathbf{r})_C = \tfrac{1}{2} \cdot r_A + 1 \cdot r_B + 0 \cdot r_C + \tfrac{1}{3} \cdot r_D = \frac{r_A}{2} + r_B + \frac{r_D}{3}$$

That is *exactly* the rule for $r_C$ from above. The transition matrix $M$ was constructed precisely so that multiplying it by $\mathbf{r}$ carries out the "sum of incoming ranks, weighted by outgoing links" calculation for every page at once. Each row of $M\mathbf{r}$ does this same calculation for a different page.

**What the equation really says.** The equation $\mathbf{r} = M\mathbf{r}$ says: when you apply the importance rule to compute every page's rank (right side), you get back the same ranks you started with (left side). The rank vector is **self-consistent**. Every page's rank already reflects the contributions flowing in from its neighbors, so applying the rule again changes nothing.

This is how linear algebra resolves the apparent circularity of "important pages are linked to by important pages." We are not defining importance in terms of itself in a vague way. We are asking for the specific vector that satisfies this fixed-point equation, and it turns out exactly one such vector exists.

Think of it like this: imagine you're repeatedly passing a rumor around a group of friends. At first, everyone is updating their version of the story. But eventually, if you keep passing it around long enough, everyone's version stabilizes and nobody's story changes anymore. That's a steady state. If you go on in mathematics, you will meet this idea more formally.



### 3.2 Starting with a Uniform Guess

One approach: start with an equal rank for every page and repeatedly multiply by $M$. This is called **power iteration**.  

Why does this work?  Multiplying by $M$ simulates one step of a random surfer clicking a link. So multiplying by
$M$ twice simulates two clicks.  When you repeat this hundreds of times, you're simulating a surfer who has been wandering the web for a very long time.

In [ ]:
# Start: every page gets the same rank (uniform distribution)
n = 4  # number of pages
r = np.array([1/n, 1/n, 1/n, 1/n])   # [0.25, 0.25, 0.25, 0.25]

print("Starting rank vector r:")
print(r)
print(f"Sum of ranks: {r.sum():.4f}   (should be 1.0)")

### 3.3 One Step of Iteration

Each step of power iteration is just one **matrix-vector multiplication**: $\mathbf{r}_{\text{new}} = M \mathbf{r}_{\text{old}}$.

Think of it this way: each page "donates" its current rank equally to all pages it links to. After one step, every page's new rank is the total amount it received.

In [ ]:
# One step of power iteration: multiply M times r
r_new = M @ r    # @ is matrix multiplication

print("After 1 step of iteration:")
print(f"  A: {r_new[0]:.4f}")
print(f"  B: {r_new[1]:.4f}")
print(f"  C: {r_new[2]:.4f}")
print(f"  D: {r_new[3]:.4f}")
print(f"Sum: {r_new.sum():.4f}   (should still be 1.0)")

### 3.4 Many Steps: Watching It Converge

If we repeat this process many times, the rank vector $\mathbf{r}$ eventually **stops changing**.  It converges to the steady state. Let's watch it happen.

The function below runs the power iteration for many steps and plots how each page's rank changes over time.

In [ ]:
# ---------------------------------------------------------------
# This function runs power iteration and plots the convergence.
# You do NOT need to modify or understand this — just run it.
# ---------------------------------------------------------------
def run_power_iteration(M, num_steps=50, page_names=None):
    """
    Runs power iteration on transition matrix M for num_steps steps.
    Returns the final rank vector and plots convergence.
    """
    n = M.shape[0]
    if page_names is None:
        page_names = [str(i) for i in range(n)]

    r = np.ones(n) / n   # start uniform
    history = [r.copy()]

    for _ in range(num_steps):
        r = M @ r
        history.append(r.copy())

    history = np.array(history)  # shape: (num_steps+1, n)

    colors = ['#4e79a7','#f28e2b','#59a14f','#e15759','#76b7b2','#edc948']
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, name in enumerate(page_names):
        ax.plot(history[:, i], label=f'Page {name}',
                color=colors[i % len(colors)], linewidth=2)
    ax.set_xlabel('Iteration (number of matrix multiplications)', fontsize=11)
    ax.set_ylabel('Rank score', fontsize=11)
    ax.set_title('Power Iteration: Rank Scores Converging to Steady State', fontsize=12)
    ax.legend()
    ax.set_ylim(-0.02, 0.7)
    ax.axhline(0, color='gray', linewidth=0.5)
    plt.tight_layout()
    plt.show()

    print("\nFinal PageRank scores (after", num_steps, "iterations):")
    for i, name in enumerate(page_names):
        print(f"  Page {name}: {r[i]:.6f}")
    print(f"  Sum: {r.sum():.6f}")
    return r

print("Function 'run_power_iteration' is defined and ready to use!")

### 3.5 Why Does This Converge? (A Peek Under the Hood)

There are really two questions here. First, why does multiplying by $M$ over and over represent something meaningful about the web? Second, why does the result settle down to a specific vector?

**Multiplication as motion.** Remember that one multiplication $M\mathbf{r}$ updates the probabilities for where the random surfer is after one click. So $M(M\mathbf{r}) = M^2 \mathbf{r}$ gives the probabilities after two clicks, $M^3 \mathbf{r}$ after three clicks, and so on. This is a key idea in linear algebra: **multiplying by a matrix twice is the composition of two linear transformations**. Each application of $M$ is one step of the random surfer, so $M^k \mathbf{r}$ tells you the probability of being on each page after $k$ clicks. Power iteration is literally simulating a very long walk through the web.

**Why it settles down. (Optional for this class, only if you are interested)** You might still wonder why this long walk approaches a *specific* vector instead of wandering forever. Here is the intuition. Any starting vector $\mathbf{r}_0$ can be written as a combination of the eigenvectors of $M$:

$$\mathbf{r}_0 = c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + c_3 \mathbf{v}_3 + c_4 \mathbf{v}_4$$

where $\mathbf{v}_1, \mathbf{v}_2, \ldots$ are the eigenvectors of $M$ with eigenvalues $\lambda_1, \lambda_2, \ldots$

When we multiply by $M$, each eigenvector gets scaled by its eigenvalue. Applying $M$ a total of $k$ times gives:

$$M^k \mathbf{r}_0 = c_1 \lambda_1^k \mathbf{v}_1 + c_2 \lambda_2^k \mathbf{v}_2 + c_3 \lambda_3^k \mathbf{v}_3 + c_4 \lambda_4^k \mathbf{v}_4$$

For a column-stochastic matrix, the largest eigenvalue is $\lambda_1 = 1$, and all the others satisfy $|\lambda_i| < 1$. So as $k$ grows:

- The $\lambda_1^k = 1^k = 1$ term stays the same.
- The other $\lambda_i^k$ terms shrink to zero, since multiplying a number smaller than 1 by itself many times sends it toward 0 (try $0.5^{20}$ on a calculator).

After many iterations, only the eigenvalue-1 piece survives:

$$M^k \mathbf{r}_0 \;\longrightarrow\; c_1 \mathbf{v}_1$$

That's why power iteration converges to the eigenvector with eigenvalue 1. The other directions literally fade away. The speed of convergence is controlled by the second-largest eigenvalue: the closer it is to 1, the slower the other terms decay, and the more iterations you need. (This is exactly what Problem 2 in the practice section is getting at.)

In [ ]:
# Run power iteration on our transition matrix M
r_converged = run_power_iteration(M, num_steps=50, page_names=['A','B','C','D'])

**Observation:** The rank scores level off and stop changing after about 20–30 iterations. The page that ends up with the highest score is the one our algorithm considers most important.

**Question to think about:** Which page ended up ranked highest? Does that make intuitive sense given the link structure of our tiny web? (Look back at the graph in Part 1.)

---

## Part 4: The Eigenvalue Connection

### 4.1 Why Is This an Eigenvector Problem?

Recall from class: a vector $\mathbf{v}$ is an **eigenvector** of a matrix $A$ with **eigenvalue** $\lambda$ if:

$$A\mathbf{v} = \lambda \mathbf{v}$$

The steady-state rank vector $\mathbf{r}$ satisfies:

$$M\mathbf{r} = \mathbf{r}$$

This is *exactly* the eigenvector equation $M\mathbf{r} = 1 \cdot \mathbf{r}$, with **eigenvalue $\lambda = 1$**!

So the PageRank vector is the **eigenvector of the transition matrix $M$ corresponding to eigenvalue 1**.

A beautiful theorem (the **Perron-Frobenius theorem**) guarantees that every column-stochastic matrix with positive entries has eigenvalue 1 as its **largest** eigenvalue, and the corresponding eigenvector has all non-negative entries. This is exactly what we need for ranks to make sense (no negative ranks!).  

### 4.2 Verifying the Steady State Is an Eigenvector

Let's verify this directly: if we multiply $M$ by our converged rank vector, we should get the same vector back.

In [ ]:
# r_converged is our steady-state rank vector from power iteration.
# Let's check: does M @ r_converged equal r_converged?

M_times_r = M @ r_converged

print("r_converged (our steady state):")
print(r_converged)
print()
print("M @ r_converged:")
print(M_times_r)
print()
print("Are they (essentially) equal?")
print(np.allclose(r_converged, M_times_r))  # allclose checks if two arrays are nearly equal
print()
print("Difference (should be nearly zero):")
print(M_times_r - r_converged)

The `True` confirms it: $M\mathbf{r} = \mathbf{r}$, so **$\mathbf{r}$ is an eigenvector of $M$ with eigenvalue 1**.

---

## Part 5: Computing PageRank Directly with Eigenvalues

### 5.1 Finding All Eigenvalues and Eigenvectors

Instead of running many iterations of power iteration, we can compute the eigenvectors of $M$ **directly** using NumPy's built-in function `np.linalg.eig()`.

This function returns:
- `eigenvalues`: an array of all eigenvalues of $M$
- `eigenvectors`: a matrix where **each column** is an eigenvector

The eigenvector we want is the one corresponding to **eigenvalue = 1**.

In [ ]:
# Compute ALL eigenvalues and eigenvectors of M at once
eigenvalues, eigenvectors = np.linalg.eig(M)

print("All eigenvalues of M:")
print(eigenvalues)
print()
print("Corresponding eigenvectors (each COLUMN is one eigenvector):")
print(eigenvectors)
print()


You should see that one of the eigenvalues is exactly (or very close to) **1.0**. The others will be smaller in magnitude. This is the guarantee from the Perron-Frobenius theorem.  (Note: Python uses $j$ where mathematicians use $i=\sqrt{-1}$ so some have complex number entries.  What is important to notice is that one of the eigenvalues is $1$.)

### 5.2 Extracting the PageRank Eigenvector

Now we need to find *which* eigenvalue is 1, grab the corresponding eigenvector column, and normalize it so the entries sum to 1.

In [ ]:
# Step 1: Find the index of the eigenvalue closest to 1
#   np.abs gives the absolute value
#   np.argmin finds the index of the minimum value
idx = np.argmin(np.abs(eigenvalues - 1.0))

print(f"The eigenvalue closest to 1 is eigenvalues[{idx}] = {eigenvalues[idx]:.6f}")
print()

# Step 2: Extract the corresponding eigenvector (that column)
raw_eigenvec = eigenvectors[:, idx]   # colon : means "all rows", idx picks the column

print("Raw eigenvector (entries may not sum to 1 yet):")
print(raw_eigenvec)
print()

# Step 3: The eigenvector might have complex-valued entries due to numerical precision.
# We take just the real part.
raw_eigenvec = raw_eigenvec.real

# Step 4: Normalize so the entries sum to 1
#   This is necessary because eigenvectors can be scaled by any constant;
#   we want ours to be a probability vector (summing to 1)
pagerank = raw_eigenvec / raw_eigenvec.sum()

print("PageRank scores (normalized eigenvector, entries sum to 1):")
page_names = ['A', 'B', 'C', 'D']
for name, score in zip(page_names, pagerank):
    print(f"  Page {name}: {score:.6f}")
print(f"\n  Sum: {pagerank.sum():.6f}")

### 5.3 Comparing the Two Methods

Let's confirm that the eigenvector method gives the **same answer** as power iteration.

In [ ]:
# Compare the two rank vectors side by side
print("Comparison: Power Iteration vs. Eigenvector Method")
print(f"{'Page':<8} {'Power Iteration':>18} {'Eigenvector Method':>20}")
print("-" * 48)
for i, name in enumerate(page_names):
    print(f"  {name:<6} {r_converged[i]:>18.6f} {pagerank[i]:>20.6f}")

print()
print("Are the two methods in agreement?")
print(np.allclose(r_converged, pagerank, atol=1e-5))

Both methods give the same PageRank scores. This is the deep connection:

> **Power iteration converges to an eigenvector. The eigenvector with eigenvalue 1 is the PageRank.**

### 5.4 Visualizing the Final Rankings

Run the cell below to see a bar chart of the PageRank scores.

In [ ]:
# Plot the PageRank scores as a bar chart
colors = ['#4e79a7','#f28e2b','#59a14f','#e15759']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(page_names, pagerank, color=colors, edgecolor='white', linewidth=1.5)

# Label each bar with its score
for bar, score in zip(bars, pagerank):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', va='bottom', fontsize=11)

ax.set_xlabel('Web Page', fontsize=12)
ax.set_ylabel('PageRank Score', fontsize=12)
ax.set_title('PageRank Scores for Our 4-Page Web', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(pagerank) * 1.25)
plt.tight_layout()
plt.show()

# Print the ranking order
ranking = sorted(zip(pagerank, page_names), reverse=True)
print("Pages ranked by importance:")
for rank_num, (score, name) in enumerate(ranking, 1):
    print(f"  #{rank_num}: Page {name}  (score = {score:.4f})")

---

## Part 6: Understanding the Results

### 6.1 Why Does Page C Rank Highest?

Look back at the link structure:
- **Every** page links to C (pages A, B, and D all point to C)
- C itself links to A, giving A a boost because it passes along its score to A
- Page D receives **no** incoming links, so it has the lowest rank
- B only receives a link from D and D has a weak score

The algorithm identifies A and C as equally important.  Notice that PageRank is not just about how many pages link to you, but how important those pages are.

### 6.2 Summary: The Math Behind PageRank

Here's the complete picture in four steps:

1. **Model the web as a directed graph** and encode it as a link matrix $L$.

2. **Normalize columns** to get the transition matrix $M$ (column-stochastic; each column sums to 1).

3. **The PageRank vector $\mathbf{r}$** is the unique vector satisfying $M\mathbf{r} = \mathbf{r}$, i.e., it is the **eigenvector of $M$ with eigenvalue 1**.

4. **Compute it** either by power iteration ($\mathbf{r} \leftarrow M\mathbf{r}$ repeated until convergence) or directly via `np.linalg.eig(M)`.



### 6.3 Why Power Iteration Wins at Scale

You might be wondering: if `np.linalg.eig` gives us the exact answer in one line, why bother with power iteration at all? For our 4-page web, the eigenvalue method is clearly the easier choice. But the real web is a different story.

**The size problem.** Google indexes hundreds of billions of pages. If we tried to write down the full transition matrix $M$ for the real web, it would have something like $10^{11}$ rows and $10^{11}$ columns. That is $10^{22}$ entries. This is a massive amount of storage just to *write down* the matrix the way we did for our toy example, let alone hand it to `np.linalg.eig`.

There is a saving grace though: most entries of $M$ are zero. A typical web page links to maybe a few dozen others, not billions. A matrix where almost all entries are zero is called **sparse**, and there are clever ways to store only the nonzero entries. This makes the storage problem manageable, but it does not solve the next problem.

**The row reduction problem.** To find an eigenvector directly, you would need to solve $(M - I)\mathbf{r} = \mathbf{0}$, which means row reducing a matrix. Row reduction is something you can do by hand on a 4-by-4 matrix, but the amount of work it takes grows as the size of the matrix gets larger. Even a supercomputer doing a billion operations per second would struggle with matrices large enough to encode the structure of the web. And row reduction has another problem: it tends to fill in those convenient zero entries with nonzero numbers as it works, so the sparseness we relied on for storage gets destroyed partway through.

**Why power iteration is the right tool.** Each step of power iteration is just one matrix-vector multiplication $M\mathbf{r}$. When $M$ is sparse, this multiplication only has to touch the nonzero entries, so each step is fast and the matrix stays sparse the whole time. We never have to modify $M$ itself. We just multiply by it, again and again, and watch the rank vector settle down. It usually takes only a few dozen iterations to get a very accurate answer.

So the tradeoff is this: `np.linalg.eig` gives you the exact answer but needs to look at the whole matrix all at once, which is impossible at web scale. Power iteration gives you an *approximate* answer, but it only ever needs to multiply by the matrix, which is something we can do efficiently even when the matrix is enormous. For a problem the size of the actual web, approximate-but-feasible beats exact-but-impossible every time.

This pattern shows up all over computational mathematics. When a problem gets too large for direct methods, **iterative methods** that approach the answer step by step often become the only practical option.

---

## Part 7: Practice Problems

Now it's your turn! For each problem, you'll make a small change to the code. Look back at the examples above if you need help.



---

### Problem 1: A Larger Web  (5 Pages)

Let's try a slightly bigger example. Here is a 5-page web (A, B, C, D, E) with the following links:

```
  A links to: B, D
  B links to: C
  C links to: A, E
  D links to: B, C, E
  E links to: A
```

**Your task:** Complete the link matrix `L5` below (some entries are already filled in), build the transition matrix, compute PageRank using eigenvalues, and print the ranking.

*Hint: Each column of L5 represents what a page links TO (which rows get a 1).*

In [ ]:
# Problem — complete the link matrix
# Rows = destination, Columns = source
# Order: A=0, B=1, C=2, D=3, E=4


# -- fill in the blank spots in the matrix

L5 = np.array([
    #  A  B  C  D  E
    [  0, 0, 1, 0, 1 ],   # row A
    [  _, _, _, _, _ ],   # row B
    [  _, _, _, _, _ ],   # row C  
    [  _, _, _ _, _ ],   # row D
    [  _, _, _, _, _ ],   # row E
], dtype=float)

# Build transition matrix
col_sums5 = L5.sum(axis=0)
M5 = L5 / col_sums5

# Compute PageRank
eigenvalues5, eigenvectors5 = np.linalg.eig(M5)
idx5 = np.argmin(np.abs(eigenvalues5 - 1.0))
raw5 = eigenvectors5[:, idx5].real
pagerank5 = raw5 / raw5.sum()

page_names5 = ['A','B','C','D','E']
print("PageRank scores for the 5-page web:")
for name, score in zip(page_names5, pagerank5):
    print(f"  Page {name}: {score:.6f}")

ranking5 = sorted(zip(pagerank5, page_names5), reverse=True)
print("\nRanked from most to least important:")
for i, (score, name) in enumerate(ranking5, 1):
    print(f"  #{i}: Page {name}  ({score:.4f})")

---

### Problem 2: How Many Iterations Until Convergence?

Power iteration converges, but how fast? The speed depends on the **second-largest eigenvalue** of $M$ (the gap between eigenvalue 1 and the next one).

**Your task:** Modify the code below to try power iteration with `num_steps = 5`, `num_steps = 15`, and `num_steps = 30` on our original matrix $M$. Run each and observe how quickly the scores stabilize.

Then answer the written question at the bottom of the cell in a comment.

In [ ]:
# Problem 2 — try different numbers of iterations

# Change the number below and re-run:
num_steps = 5    # try 5, then 15, then 30

r_test = run_power_iteration(M, num_steps=num_steps, page_names=['A','B','C','D'])

print()
print("Exact PageRank from eigenvector method (for comparison):")
for name, score in zip(page_names, pagerank):
    print(f"  Page {name}: {score:.6f}")



### Type your response here (double click to edit the cell).  After how many iterations do the scores look "close enough" to the exact answer (within 3 decimal places)?

---

## Reflection

Before you finish, think about these questions. You don't need to write code, just write your response in the cell below.

1. **The eigenvector equation** $M\mathbf{r} = \mathbf{r}$ looks simple. Why does it make sense as a definition of "importance"? (Hint: think about what happens to the rank vector when it's already at steady state.)


2. **What would happen** if a page had no outgoing links at all (a "dangling node")? Imagine our random surfer lands on such a page. With no links to click, what should happen next? And what does this mean for our model, where each step is just multiplying by $M$? Where does the rank that flowed into that page end up?


3. **Scale:** Google's web has hundreds of billions of pages. We can't store a $10^{11} \times 10^{11}$ matrix! How do you think power iteration is useful here, even if computing all eigenvectors directly is impossible?

### Type reflections here (double-click to edit this cell):

1. 

2. 

3. 

---

## Summary

Congratulations — you've just implemented and understood the core mathematics behind one of the most influential algorithms ever written! Here's what you covered:

| Concept | What we did |
|---|---|
| **Directed graphs** | Modeled the web as a matrix |
| **Column-stochastic matrices** | Normalized the link matrix so columns sum to 1 |
| **Matrix-vector multiplication** | Each step of power iteration is one `M @ r` |
| **Convergence** | Repeated multiplication drives `r` to a steady state |
| **Eigenvectors** | The steady state satisfies $M\mathbf{r} = \mathbf{r}$ (eigenvalue 1) |
| **`np.linalg.eig`** | Computed the exact PageRank directly |

The connection between **repeated matrix multiplication** and **eigenvectors** is one of the most powerful ideas in applied mathematics. You might see it again in future classes in topics like Markov chains, principal component analysis (PCA), and quantum mechanics.

---
To submit your lab, after running all cells, export it as a pdf and upload to Canvas.